### install packages

In [ ]:
pip install pandas numpy matplotlib polars pyarrow plotly nbformat

In [29]:
pip install -U kaleido

Python(85832) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 66 kB 7.2 MB/s eta 0:00:011
     |████████████████████████████████| 56 kB 25.6 MB/s eta 0:00:01
     |████████████████████████████████| 243 kB 40.7 MB/s eta 0:00:01
     |████████████████████████████████| 76 kB 26.0 MB/s eta 0:00:01
     |████████████████████████████████| 365 kB 86.8 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
import polars as pl
import gc
import pyarrow
from pathlib import Path #to keep importing datasets from relative paths consistent across OS
import plotly
import plotly.express as px
import nbformat
import plotly.io as pio
import kaleido


### define path

In [2]:
#Get current notebook directory
current_dir = Path(__file__).parent if "__file__" in globals() else Path.cwd()

# Define path relative to project root
data_path = current_dir.parent.parent / "data" 


## read data

In [ ]:
df = pl.read_csv(data_path / 'random_state158.csv')

### data cleaning

In [4]:
# userId_y and timestamp_y come from tags and should be dropped
# correct userId_x and timestamp_x names
df = df.drop(["userId_y","timestamp_y"])


In [5]:
df = df.rename({
    'userId_x' : 'userId',
    'timestamp_x' : 'timestamp'
})

In [6]:
df.head()

userId,movieId,rating,timestamp,tag,title,genres,imdbId,tmdbId
i64,i64,f64,i64,str,str,str,i64,f64
17682,31433,3.5,1544094795,"""hot guys""","""Wedding Date, The (2005)""","""Comedy|Romance""",372532,6961.0
17682,31433,3.5,1544094795,"""family""","""Wedding Date, The (2005)""","""Comedy|Romance""",372532,6961.0
17682,31433,3.5,1544094795,"""romance""","""Wedding Date, The (2005)""","""Comedy|Romance""",372532,6961.0
17682,31433,3.5,1544094795,"""watched 2007""","""Wedding Date, The (2005)""","""Comedy|Romance""",372532,6961.0
17682,31433,3.5,1544094795,"""Debra Messing""","""Wedding Date, The (2005)""","""Comedy|Romance""",372532,6961.0


In [7]:
df.shape

(103531029, 9)

### statistics: totals

In [8]:

# Count unique userId and movieId
unique_userId = df["userId"].n_unique()
unique_movieId = df["movieId"].n_unique()

print(unique_userId, unique_movieId)



75598 12301


## Remove "tag" column for fast summaries

In [9]:
# 1. Drop the 'tag' column if it exists
if "tag" in df.columns:
    df = df.drop("tag")

# 2. Remove duplicate rows
df = df.unique()


In [10]:
gc.collect()
df.shape

(149177, 8)

### convert to pandas df

In [11]:
df = df.to_pandas()

In [14]:
# write to csv for future use
df.to_csv(data_path / 'sample_no_tags.csv')

In [3]:
### read no tags csv sample if needed
df = pd.read_csv(data_path / "sample_no_tags.csv")

### statistics

In [ ]:
# show plots in browser
pio.renderers.default = "browser"  # change to notebook if you want in line in notebook

In [ ]:

#4a. Distribution of unique userId per movieId
user_per_movie = df.groupby('movieId')['userId'].nunique().reset_index(name='unique_user_count')
fig1 = px.histogram(user_per_movie, x='unique_user_count', nbins=50, title='Unique userId per movieId')
fig1.show()


In [6]:

# 4b. Distribution of unique movieId per userId
movie_per_user = df.groupby('userId')['movieId'].nunique().reset_index(name='unique_movie_count')
fig2 = px.histogram(movie_per_user, x='unique_movie_count', nbins=50, title='Unique movieId per userId')
fig2.show()


In [8]:

# Calculate each user's average rating
user_avg_rating = df.groupby("userId")["rating"].mean().reset_index(name="avg_rating")

# Plot the distribution
fig = px.histogram(user_avg_rating, x="avg_rating", nbins=10,
                   title="Distribution of Each User's Average Ratings")
fig.show()
